In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, mean_squared_error,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from xgboost.sklearn import XGBClassifier


In [128]:
DATA_PATH = "../flights_data/"

TEST_FILENAME_PATH = DATA_PATH + "flight_delays_test.csv"
TRAIN_FILENAME_PATH = DATA_PATH + "flight_delays_train.csv"

In [129]:
train = pd.read_csv(TRAIN_FILENAME_PATH)
test = pd.read_csv(TEST_FILENAME_PATH)
train

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732,N
1,c-4,c-20,c-3,1548,US,PIT,MCO,834,N
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416,N
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872,N
4,c-10,c-7,c-6,1828,WN,MDW,OMA,423,Y
...,...,...,...,...,...,...,...,...,...
99995,c-5,c-4,c-3,1618,OO,SFO,RDD,199,N
99996,c-1,c-18,c-3,804,CO,EWR,DAB,884,N
99997,c-1,c-24,c-2,1901,NW,DTW,IAH,1076,N
99998,c-4,c-27,c-4,1515,MQ,DFW,GGG,140,N


In [130]:
train['Month'] = train['Month'].str[2:].astype('int')
train['DayofMonth'] = train['DayofMonth'].str[2:].astype('int')
train['DayOfWeek'] = train['DayOfWeek'].str[2:].astype('int')
train['DepTime_hour'] =  train['DepTime'] // 100
train['DepTime_minute'] =  train['DepTime'] % 100

test['Month'] = test['Month'].str[2:].astype('int')
test['DayofMonth'] = test['DayofMonth'].str[2:].astype('int')
test['DayOfWeek'] = test['DayOfWeek'].str[2:].astype('int')
test['DepTime_hour'] =  test['DepTime'] // 100
test['DepTime_minute'] =  test['DepTime'] % 100
train['s_hour'] = np.sin(2*np.pi*train['DepTime_hour'] / 24)
train['c_hour'] = np.cos(2*np.pi*train['DepTime_hour'] / 24)

test['s_hour'] = np.sin(2*np.pi*test['DepTime_hour']/24)
test['c_hour'] = np.cos(2*np.pi*test['DepTime_hour']/24)
train['morning'] = ((train['DepTime_hour'] >= 6) & (train['DepTime_hour'] < 12)).astype('int')
train['day'] = ((train['DepTime_hour'] >= 12) & (train['DepTime_hour'] < 18)).astype('int')
train['evening'] = ((train['DepTime_hour'] >= 18) & (train['DepTime_hour'] < 24)).astype('int')
train['night'] = ((train['DepTime_hour'] >= 0) & (train['DepTime_hour'] < 6)).astype('int')

train['low_delay'] = ((train['DepTime_hour'] >= 4) & (train['DepTime_hour'] < 9)).astype('int')
train['high_delay'] = ((train['DepTime_hour'] >= 4) & (train['DepTime_hour'] < 9)).astype('int')

test['morning'] = ((test['DepTime_hour'] >= 6) & (test['DepTime_hour'] < 12)).astype('int')
test['day'] = ((test['DepTime_hour'] >= 12) & (test['DepTime_hour'] < 18)).astype('int')
test['evening'] = ((test['DepTime_hour'] >= 18) & (test['DepTime_hour'] < 24)).astype('int')
test['night'] = ((test['DepTime_hour'] >= 0) & (test['DepTime_hour'] < 6)).astype('int')
test['low_delay'] = ((test['DepTime_hour'] >= 4) & (test['DepTime_hour'] < 9)).astype('int')
test['other_time'] = ((test['DepTime_hour'] < 9) & (test['DepTime_hour'] >=5)).astype('int')
test['delay_time'] = ((test['DepTime_hour'] >= 13) & (test['DepTime_hour'] < 24) | (test['DepTime_hour'] < 5)).astype('int')
test['middle_time'] = ((test['DepTime_hour'] >= 9) & (test['DepTime_hour'] < 13)).astype('int')
train['delay_time'] = ((train['DepTime_hour'] >= 13) & (train['DepTime_hour'] < 24) | (train['DepTime_hour'] < 5)).astype('int')
train['other_time'] = ((train['DepTime_hour'] < 9) & (train['DepTime_hour'] >=5)).astype('int')
train['middle_time'] = ((train['DepTime_hour'] >= 9) & (train['DepTime_hour'] < 13)).astype('int')
train['month_x'] = ((train['Month'].isin([12, 6, 7]))).astype('int')
test['month_x'] = ((test['Month'].isin([12, 1]))).astype('int')

train['month_y'] = ((train['Month'].isin([4, 5, 9, 2]))).astype('int')
test['month_y'] = ((test['Month'].isin([12, 1]))).astype('int')
train['winter'] = ((train['Month'].isin([12, 1, 2]))).astype('int')
test['winter'] = ((test['Month'].isin([12, 1, 2]))).astype('int')

train['spring'] = ((train['Month'].isin([3, 4, 5]))).astype('int')
test['spring'] = ((test['Month'].isin([3, 4, 5]))).astype('int')

train['summer'] = ((train['Month'].isin([6, 7, 8]))).astype('int')
test['summer'] = ((test['Month'].isin([6, 7, 8]))).astype('int')

train['autumn'] = ((train['Month'].isin([9, 10, 11]))).astype('int')
test['autumn'] = ((test['Month'].isin([9, 10, 11]))).astype('int')
train['week_high'] = ((train['DayOfWeek'].isin([4, 5, 1, 7]))).astype('int')
test['week_high'] = ((test['DayOfWeek'].isin([12, 1, 2]))).astype('int')

train['week_low'] = ((train['DayOfWeek'].isin([6, 2, 3]))).astype('int')
test['week_low'] = ((test['DayOfWeek'].isin([12, 1, 2]))).astype('int')


In [131]:
train

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min,DepTime_hour,...,other_time,middle_time,month_x,month_y,winter,spring,summer,autumn,week_high,week_low
0,8,21,7,1934,AA,ATL,DFW,732,N,19,...,0,0,0,0,0,0,1,0,1,0
1,4,20,3,1548,US,PIT,MCO,834,N,15,...,0,0,0,1,0,1,0,0,0,1
2,9,2,5,1422,XE,RDU,CLE,416,N,14,...,0,0,0,1,0,0,0,1,1,0
3,11,25,6,1015,OO,DEN,MEM,872,N,10,...,0,1,0,0,0,0,0,1,0,1
4,10,7,6,1828,WN,MDW,OMA,423,Y,18,...,0,0,0,0,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,5,4,3,1618,OO,SFO,RDD,199,N,16,...,0,0,0,1,0,1,0,0,0,1
99996,1,18,3,804,CO,EWR,DAB,884,N,8,...,1,0,0,0,1,0,0,0,0,1
99997,1,24,2,1901,NW,DTW,IAH,1076,N,19,...,0,0,0,0,1,0,0,0,0,1
99998,4,27,4,1515,MQ,DFW,GGG,140,N,15,...,0,0,0,1,0,1,0,0,1,0


Берем не все фичи

In [126]:
cols = ['Month', 'DayofMonth', 'DayOfWeek', 's_hour', 'c_hour', 'Distance', ]
X_train = train[cols + ['dep_delayed_15min']]
X_test = test[cols]
X_train

,Month,DayofMonth,DayOfWeek,s_hour,c_hour,Distance,dep_delayed_15min
0,8,21,7,-0.965926,2.588190e-01,732,N
1,4,20,3,-0.707107,-7.071068e-01,834,N
2,9,2,5,-0.500000,-8.660254e-01,416,N
3,11,25,6,0.500000,-8.660254e-01,872,N
4,10,7,6,-1.000000,-1.836970e-16,423,Y
...,...,...,...,...,...,...,...
99995,5,4,3,-0.866025,-5.000000e-01,199,N
99996,1,18,3,0.866025,-5.000000e-01,884,N
99997,1,24,2,-0.965926,2.588190e-01,1076,N
99998,4,27,4,-0.707107,-7.071068e-01,140,N


Перезапишем датасеты и закоммитим их:

In [132]:
X_train.to_csv(TRAIN_FILENAME_PATH, index=False) 
X_test.to_csv(TEST_FILENAME_PATH, index=False) 

Здесь же обучил модели, поскольку набор фич другой

In [134]:
print("Запуск обучения")

cols = ['Month', 'DayofMonth', 'DayOfWeek', 's_hour', 'c_hour', 'Distance']
X_train = train[cols]
X_test = test[cols]
y_train = y


np.random.seed(51)
X_train_part, X_valid, y_train_part, y_valid = train_test_split(
    X_train, y_train, test_size=0.3
)

print("Обучение модели")

xgb_model = XGBClassifier()
xgb_model.fit(X_train_part, y_train_part)

preds = xgb_model.predict(X_valid)
preds_probas = xgb_model.predict_proba(X_valid)[:, 1]
print("Результат обучения:")


print("Accuracy:", accuracy_score(y_valid, preds))
print("Precision:", precision_score(y_valid, preds))
print("Recall:", recall_score(y_valid, preds))
print("MSE:", mean_squared_error(y_valid, preds))
print("ROC-AUC:", roc_auc_score(y_valid, preds_probas))

xgb_model.save_model(DATA_PATH + "xgb_model.json")

Запуск обучения
Обучение модели
Результат обучения:
Accuracy: 0.8148
Precision: 0.5601374570446735
Recall: 0.11446629213483146
MSE: 0.1852
ROC-AUC: 0.7104069897231048
